In [3]:
# ── QUICK RELOAD (run this if kernel restarts) ──────────────────
import pandas as pd
import os

BASE_DIR = os.path.expanduser("~/projects/nhs-ae-waiting-times")
BRONZE_DIR = os.path.join(BASE_DIR, "data/bronze")
SILVER_DIR = os.path.join(BASE_DIR, "data/silver")

# Load already-saved Silver back into memory instantly
df = pd.read_parquet(os.path.join(SILVER_DIR, "silver_ae_providers.parquet"))
silver_totals = pd.read_parquet(os.path.join(SILVER_DIR, "silver_ae_national_totals.parquet"))

print(f"Silver loaded: {df.shape}")
print(f"National totals loaded: {len(silver_totals)} rows")

Silver loaded: (7254, 25)
National totals loaded: 20 rows


In [1]:
import pandas as pd
import os
import glob
from datetime import datetime

BASE_DIR = os.path.expanduser("~/projects/nhs-ae-waiting-times")
BRONZE_DIR = os.path.join(BASE_DIR, "data/bronze")
SILVER_DIR = os.path.join(BASE_DIR, "data/silver")

print("Bronze dir:", BRONZE_DIR)
print("Silver dir:", SILVER_DIR)

Bronze dir: /Users/yusufismail/projects/nhs-ae-waiting-times/data/bronze
Silver dir: /Users/yusufismail/projects/nhs-ae-waiting-times/data/silver


In [3]:
# Grab every parquet file across all three year folders
all_files = glob.glob(os.path.join(BRONZE_DIR, "**/*.parquet"), recursive=True)
all_files = sorted(all_files)

print(f"Found {len(all_files)} Bronze files\n")

# Load and combine
dfs = []
for f in all_files:
    df = pd.read_parquet(f)
    dfs.append(df)

raw = pd.concat(dfs, ignore_index=True)

print(f"Combined shape: {raw.shape}")
print(f"Columns: {raw.columns.tolist()}")

Found 36 Bronze files

Combined shape: (7274, 31)
Columns: ['Period', 'Org Code', 'Parent Org', 'Org name', 'A&E attendances Type 1', 'A&E attendances Type 2', 'A&E attendances Other A&E Department', 'A&E attendances Booked Appointments Type 1', 'A&E attendances Booked Appointments Type 2', 'A&E attendances Booked Appointments Other Department', 'Attendances over 4hrs Type 1', 'Attendances over 4hrs Type 2', 'Attendances over 4hrs Other Department', 'Attendances over 4hrs Booked Appointments Type 1', 'Attendances over 4hrs Booked Appointments Type 2', 'Attendances over 4hrs Booked Appointments Other Department', 'Patients who have waited 4-12 hs from DTA to admission', 'Patients who have waited 12+ hrs from DTA to admission', 'Emergency admissions via A&E - Type 1', 'Emergency admissions via A&E - Type 2', 'Emergency admissions via A&E - Other A&E department', 'Other emergency admissions', '_ingestion_timestamp', '_source_url', '_source_file', 'Unnamed: 22', 'Unnamed: 23', 'Unnamed: 24

In [5]:
# Check column counts per source file to understand the schema difference
col_counts = {}
for f in all_files:
    df = pd.read_parquet(f)
    col_counts[os.path.basename(f)] = len(df.columns)

anomalies = {k: v for k, v in col_counts.items() if v != 25}
print("Files with non-standard column counts:")
for k, v in anomalies.items():
    print(f"  {k}: {v} columns")

# Show what the extra columns are in September 2024
sep_file = [f for f in all_files if 'september_2024' in f][0]
sep_df = pd.read_parquet(sep_file)
standard_file = [f for f in all_files if 'august_2024' in f][0]
std_df = pd.read_parquet(standard_file)

extra_cols = set(sep_df.columns) - set(std_df.columns)
print(f"\nExtra columns in September 2024:")
for col in extra_cols:
    print(f"  '{col}'")

Files with non-standard column counts:
  monthly_ae_september_2024.parquet: 31 columns

Extra columns in September 2024:
  'Unnamed: 26'
  'Unnamed: 25'
  'Unnamed: 24'
  'Unnamed: 23'
  'a'
  'Unnamed: 22'


In [7]:
# ── SILVER TRANSFORMATION ──────────────────────────────────────────

# Step 1: Drop junk columns from September 2024 anomaly
junk_cols = [c for c in raw.columns if str(c).startswith('Unnamed') or str(c) == 'a']
df = raw.drop(columns=junk_cols)
print(f"Step 1 — Dropped {len(junk_cols)} junk columns: {junk_cols}")
print(f"Shape after drop: {df.shape}")

# Step 2: Rename columns to clean snake_case
column_mapping = {
    'Period': 'period_raw',
    'Org Code': 'org_code',
    'Parent Org': 'parent_org',
    'Org name': 'org_name',
    'A&E attendances Type 1': 'att_type1',
    'A&E attendances Type 2': 'att_type2',
    'A&E attendances Other A&E Department': 'att_other',
    'A&E attendances Booked Appointments Type 1': 'att_booked_type1',
    'A&E attendances Booked Appointments Type 2': 'att_booked_type2',
    'A&E attendances Booked Appointments Other Department': 'att_booked_other',
    'Attendances over 4hrs Type 1': 'over_4hr_type1',
    'Attendances over 4hrs Type 2': 'over_4hr_type2',
    'Attendances over 4hrs Other Department': 'over_4hr_other',
    'Attendances over 4hrs Booked Appointments Type 1': 'over_4hr_booked_type1',
    'Attendances over 4hrs Booked Appointments Type 2': 'over_4hr_booked_type2',
    'Attendances over 4hrs Booked Appointments Other Department': 'over_4hr_booked_other',
    'Patients who have waited 4-12 hs from DTA to admission': 'dta_4_12hr',
    'Patients who have waited 12+ hrs from DTA to admission': 'dta_12hr_plus',
    'Emergency admissions via A&E - Type 1': 'emerg_admit_type1',
    'Emergency admissions via A&E - Type 2': 'emerg_admit_type2',
    'Emergency admissions via A&E - Other A&E department': 'emerg_admit_other',
    'Other emergency admissions': 'emerg_admit_non_ae',
}
df = df.rename(columns=column_mapping)
print(f"\nStep 2 — Columns renamed")

# Step 3: Separate TOTAL rows from provider rows
silver_totals = df[df['period_raw'] == 'TOTAL'].copy()
df = df[df['period_raw'] != 'TOTAL'].copy()
print(f"\nStep 3 — Separated TOTAL rows")
print(f"  Provider rows: {len(df)}")
print(f"  TOTAL rows: {len(silver_totals)}")

# Step 4: Parse period_raw into a proper date
# 'MSitAE-MARCH-2025' → 2025-03-01
month_map = {
    'JANUARY': '01', 'FEBRUARY': '02', 'MARCH': '03', 'APRIL': '04',
    'MAY': '05', 'JUNE': '06', 'JULY': '07', 'AUGUST': '08',
    'SEPTEMBER': '09', 'OCTOBER': '10', 'NOVEMBER': '11', 'DECEMBER': '12'
}

def parse_period(period_str):
    try:
        parts = period_str.split('-')  # ['MSitAE', 'MARCH', '2025']
        month = month_map[parts[1]]
        year = parts[2]
        return pd.to_datetime(f"{year}-{month}-01")
    except:
        return pd.NaT

df['period_date'] = df['period_raw'].apply(parse_period)
print(f"\nStep 4 — Period parsed")
print(f"  Date range: {df['period_date'].min()} → {df['period_date'].max()}")
print(f"  Null dates: {df['period_date'].isna().sum()}")

# Step 5: Clean org fields
df['org_code'] = df['org_code'].str.strip()
df['org_name'] = df['org_name'].str.strip()
df['parent_org'] = df['parent_org'].str.strip()
print(f"\nStep 5 — Org fields stripped")

# Step 6: Classify provider type
def classify_provider(org_code, org_name):
    org_code = str(org_code).strip()
    org_name = str(org_name).upper()
    
    # Standard NHS Trust ODS codes: 3 chars starting with R or a digit
    if len(org_code) == 3 and org_code[0] in ('R', 'r'):
        return 'NHS_TRUST'
    # GP practice codes: start with letter + 5 digits
    elif len(org_code) == 6 and org_code[0].isalpha() and org_code[1:].isdigit():
        return 'GP_PRACTICE'
    # Y-codes = ICB/commissioned services
    elif org_code.startswith('Y'):
        return 'COMMISSIONED_SERVICE'
    # Independent sector indicators
    elif any(word in org_name for word in ['LTD', 'LIMITED', 'PLC', 'ASSURA', 'VOCARE', 'BADGER']):
        return 'INDEPENDENT'
    else:
        return 'OTHER'

df['provider_type'] = df.apply(
    lambda row: classify_provider(row['org_code'], row['org_name']), axis=1
)
print(f"\nStep 6 — Provider types classified:")
print(df['provider_type'].value_counts().to_string())

# Step 7: Add has_type1_ae flag
df['has_type1_ae'] = df['att_type1'] > 0
print(f"\nStep 7 — Type 1 A&E flag added")
print(f"  Providers with Type 1 A&E: {df['has_type1_ae'].sum()} rows")
print(f"  Providers without Type 1 A&E: {(~df['has_type1_ae']).sum()} rows")

# Step 8: Drop audit columns (they stay in Bronze, not needed in Silver)
df = df.drop(columns=['_ingestion_timestamp', '_source_url', '_source_file'])
silver_totals = silver_totals.drop(columns=['_ingestion_timestamp', '_source_url', '_source_file'])

print(f"\nStep 8 — Audit columns dropped")
print(f"\nFinal Silver shape: {df.shape}")
print(f"Final Silver columns: {df.columns.tolist()}")

Step 1 — Dropped 6 junk columns: ['Unnamed: 22', 'Unnamed: 23', 'Unnamed: 24', 'Unnamed: 25', 'Unnamed: 26', 'a']
Shape after drop: (7274, 25)

Step 2 — Columns renamed

Step 3 — Separated TOTAL rows
  Provider rows: 7254
  TOTAL rows: 20

Step 4 — Period parsed
  Date range: 2022-04-01 00:00:00 → 2025-03-01 00:00:00
  Null dates: 16

Step 5 — Org fields stripped

Step 6 — Provider types classified:
provider_type
NHS_TRUST      5457
OTHER          1140
GP_PRACTICE     502
INDEPENDENT     155

Step 7 — Type 1 A&E flag added
  Providers with Type 1 A&E: 4432 rows
  Providers without Type 1 A&E: 2822 rows

Step 8 — Audit columns dropped

Final Silver shape: (7254, 25)
Final Silver columns: ['period_raw', 'org_code', 'parent_org', 'org_name', 'att_type1', 'att_type2', 'att_other', 'att_booked_type1', 'att_booked_type2', 'att_booked_other', 'over_4hr_type1', 'over_4hr_type2', 'over_4hr_other', 'over_4hr_booked_type1', 'over_4hr_booked_type2', 'over_4hr_booked_other', 'dta_4_12hr', 'dta_12hr

In [9]:
# Save main provider Silver table
silver_path = os.path.join(SILVER_DIR, "silver_ae_providers.parquet")
df.to_parquet(silver_path, index=False)

# Save national totals separately
totals_path = os.path.join(SILVER_DIR, "silver_ae_national_totals.parquet")
silver_totals.to_parquet(totals_path, index=False)

print(f"✅ Silver provider table saved — {len(df):,} rows")
print(f"✅ Silver national totals saved — {len(silver_totals):,} rows")

# Verify
silver_size = os.path.getsize(silver_path) / 1024
print(f"\nSilver file size: {silver_size:.1f} KB")

✅ Silver provider table saved — 7,254 rows
✅ Silver national totals saved — 20 rows

Silver file size: 253.2 KB


In [5]:
# Investigate null dates
print("=== Rows with null period_date ===")
null_dates = df[df['period_date'].isna()]
print(f"Count: {len(null_dates)}")
print(null_dates[['period_raw', 'org_code', 'org_name']].to_string())

print("\n=== Sample of OTHER provider types ===")
other_providers = df[df['provider_type'] == 'OTHER']
print(f"Count: {len(other_providers)}")
print(other_providers[['org_code', 'org_name', 'parent_org']].drop_duplicates('org_code').head(20).to_string())

=== Rows with null period_date ===
Count: 16
     period_raw org_code org_name
204       Total    Total    Total
2647      TOTAl    TOTAl    TOTAl
2851      Total    Total    Total
3054      Total    Total    Total
3254      Total    Total    Total
3454      Total    Total    Total
3658      Total    Total    Total
3863      Total    Total    Total
4061      Total    Total    Total
4266      Total    Total    Total
4470      Total    Total    Total
4674      Total    Total    Total
4878     Total     Total    Total
5076     Total     Total    Total
6067      Total    Total    Total
7253      Total    Total    Total

=== Sample of OTHER provider types ===
Count: 1140
   org_code                                             org_name                            parent_org
1     AD913                                 BECKENHAM BEACON UCC                    NHS ENGLAND LONDON
5     NJS05                      LOUGHBOROUGH URGENT CARE CENTRE                  NHS ENGLAND MIDLANDS
7     NNJ0H  LLR

In [7]:
# ── FIXES TO SILVER ─────────────────────────────────────────────

# Fix 1: Remove ALL Total rows (case-insensitive)
df = df[~df['period_raw'].str.upper().isin(['TOTAL'])].copy()
df = df[~df['org_code'].str.upper().isin(['TOTAL'])].copy()

print(f"After removing all Total rows: {len(df)} rows")
print(f"Null dates remaining: {df['period_date'].isna().sum()}")

# Fix 2: Improved provider classifier
def classify_provider_v2(org_code, org_name):
    org_code = str(org_code).strip()
    org_name = str(org_name).upper()

    # Standard NHS Trust ODS codes: 3 chars starting with R
    if len(org_code) == 3 and org_code[0].upper() == 'R':
        return 'NHS_TRUST'
    
    # GP practice codes: 1 letter + 5 digits (e.g. C82009)
    elif len(org_code) == 6 and org_code[0].isalpha() and org_code[1:].isdigit():
        return 'GP_PRACTICE'
    
    # Y-codes = ICB commissioned services
    elif org_code.startswith('Y'):
        return 'COMMISSIONED_SERVICE'
    
    # NN*, NQ*, NT*, NP* = UTC/community/urgent care ODS codes
    elif len(org_code) >= 3 and org_code[:2] in ('NN', 'NQ', 'NT', 'NP', 'NL', 'NR', 'NJ'):
        return 'UTC_COMMUNITY'
    
    # A-codes and other alphanumeric = independent/commissioned urgent care
    elif len(org_code) >= 3 and org_code[0] in ('A', 'D') and any(c.isdigit() for c in org_code):
        return 'INDEPENDENT'
    
    # Named independent sector indicators
    elif any(word in org_name for word in ['LTD', 'LIMITED', 'PLC', 'ASSURA', 'VOCARE', 
                                            'BADGER', 'PRACTICE PLUS', 'DHU']):
        return 'INDEPENDENT'
    
    # 3-char codes not starting with R = other NHS bodies
    elif len(org_code) == 3:
        return 'NHS_OTHER'
    
    else:
        return 'OTHER'

df['provider_type'] = df.apply(
    lambda row: classify_provider_v2(row['org_code'], row['org_name']), axis=1
)

print(f"\nUpdated provider type breakdown:")
print(df['provider_type'].value_counts().to_string())

print(f"\nRemaining OTHER count: {(df['provider_type'] == 'OTHER').sum()}")
print("\nRemaining OTHER sample:")
print(df[df['provider_type'] == 'OTHER'][['org_code', 'org_name']].drop_duplicates().head(10).to_string())

After removing all Total rows: 7238 rows
Null dates remaining: 0

Updated provider type breakdown:
provider_type
NHS_TRUST        5457
UTC_COMMUNITY     798
GP_PRACTICE       502
INDEPENDENT       307
OTHER             103
NHS_OTHER          71

Remaining OTHER count: 103

Remaining OTHER sample:
     org_code                       org_name
125     NDA57  HASLEMERE MINOR INJURIES UNIT
304     NV693           ROSSENDALE MIU & OOH
2645    G0Q0L           BARKING HOSPITAL UTC
2646    K6K0R     HAROLD WOOD POLYCLINIC UTC


In [11]:
# Fix 3: Relabel remaining OTHER as UTC_COMMUNITY — they're all minor injury/UTC providers
df['provider_type'] = df['provider_type'].replace('OTHER', 'UTC_COMMUNITY')

print("Final provider type breakdown:")
print(df['provider_type'].value_counts().to_string())
print(f"\nAny OTHER remaining: {(df['provider_type'] == 'OTHER').sum()}")
print(f"\nFinal Silver shape: {df.shape}")

# Resave the corrected Silver file
silver_path = os.path.join(SILVER_DIR, "silver_ae_providers.parquet")
df.to_parquet(silver_path, index=False)
print(f"\n✅ Silver file resaved — {len(df):,} rows, {df.shape[1]} columns")
print(f"File size: {os.path.getsize(silver_path)/1024:.1f} KB")

Final provider type breakdown:
provider_type
NHS_TRUST        5457
UTC_COMMUNITY     901
GP_PRACTICE       502
INDEPENDENT       307
NHS_OTHER          71

Any OTHER remaining: 0

Final Silver shape: (7238, 25)

✅ Silver file resaved — 7,238 rows, 25 columns
File size: 250.4 KB
